# Companion Notebook — *The Universal Folding Theorem*

This notebook is a runnable companion to **"The Universal Folding Theorem: Ontological Isomorphisms Across Cryptographic, Biological, and Evolutionary Search Spaces"**.

It is designed to do four things:

1. restate the paper's core runtime in executable form,
2. reproduce the exact quantitative pieces that are already explicit in the paper,
3. build transparent toy experiments for the paper's operational claims, and
4. keep the proof boundary clean between **exact formulas from the paper** and **surrogate models used for intuition**.

## Proof boundary

This notebook treats the following as **paper-level claims stated explicitly in the manuscript**:

- a universal search tuple with state space, variation, projection, and selection,
- the thermodynamic impossibility of flat exhaustive search on very large spaces,
- the Mark 1 attractor
  $$
  H = \frac{\pi}{9},
  $$
- the local curvature-loss approximation
  $$
  \epsilon(\theta) \approx \frac{\theta^2}{24},
  $$
  hence
  $$
  \epsilon(H) = \frac{H^2}{24},
  $$
- the operational sequence variation $\to$ projection $\to$ selection $\to$ retained survivor,
- the role of AHRC as a convergence controller rather than the fold itself.

This notebook treats the following as **transparent computational surrogates**:

- toy folding landscapes,
- toy evolutionary search,
- simplified nonce search experiments,
- a minimal AHRC-like controller implemented numerically.

Those surrogates are not presented as proofs of the full paper. They are executable models for the paper's logic.

In [ ]:
# Install cell for a fresh environment
# Run this once if needed.
%pip -q install numpy pandas matplotlib

In [ ]:
import math
import hashlib
import random
from dataclasses import dataclass
from typing import Callable, Any, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
random.seed(7)
np.random.seed(7)

## 1. Core constants reproduced from the paper

The paper explicitly uses the Mark 1 attractor

$$
H = \frac{\pi}{9},
$$

and the local chord–arc curvature-loss approximation

$$
\epsilon(\theta) \approx \frac{\theta^2}{24}.
$$

At $\theta = H$, this gives the paper's local interface-tension estimate.

In [ ]:
H = math.pi / 9
epsilon_H = H**2 / 24

constants_df = pd.DataFrame(
    {
        "quantity": ["H", "epsilon(H)", "degrees(H)", "circle closure steps"],
        "value": [H, epsilon_H, math.degrees(H), 2*math.pi / H],
    }
)
constants_df

## 2. Universal search tuple

The paper formalizes a universal search runtime over a candidate space with a variation operator, a projection map, and a selection / stability functional, then retains the surviving state lineage.

In compact executable form we can represent the runtime as:

$$
x_{t+1} = \operatorname{Retain}\bigl(\mathcal{S}(\mathcal{P}(\mathcal{V}(x_t)))\bigr),
$$

where:

- $\mathcal{V}$ = variation,
- $\mathcal{P}$ = projection / rendered face,
- $\mathcal{S}$ = selection / fitness audit,
- retain = survivor persistence.

In [ ]:
@dataclass
class UniversalRuntime:
    variation: Callable[[Any], Any]
    projection: Callable[[Any], Any]
    selection: Callable[[Any], float]
    retain_rule: Callable[[Any, Any, float], Any]

    def step(self, x: Any) -> Dict[str, Any]:
        y = self.variation(x)
        face = self.projection(y)
        score = self.selection(face)
        survivor = self.retain_rule(x, y, score)
        return {
            "input_state": x,
            "varied_state": y,
            "projected_face": face,
            "selection_score": score,
            "retained_state": survivor,
        }

## 3. Thermodynamic lower bound on flat exhaustive search

The paper's thermodynamic claim is simple and load-bearing:

- each evaluation has positive cost,
- the state space is enormous,
- if the flat search cost exceeds the available budget, exhaustive search is impossible in practice and the runtime must exploit structure.

We summarize this as

$$
C_{\text{flat}} \ge N \cdot c_{\text{eval}},
$$

and exhaustive search is prohibited whenever

$$
C_{\text{flat}} > R,
$$

where $R$ is the resource budget.

In [ ]:
resource_examples = pd.DataFrame(
    [
        ("Protein toy (100 residues, Levinthal scale)", 1e95),
        ("SHA-256 output space", 2**256),
        ("Bitstring space, n=64", 2**64),
        ("Bitstring space, n=24", 2**24),
    ],
    columns=["system", "state_space_size"],
)
resource_examples["log10_state_space"] = np.log10(resource_examples["state_space_size"].astype(float))
resource_examples

In [ ]:
fig, ax = plt.subplots()
ax.bar(resource_examples["system"], resource_examples["log10_state_space"])
ax.set_ylabel(r"$\log_{10}(N)$")
ax.set_title("Log-scale candidate state spaces")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 4. Blind search as the degenerate limit

For a Bernoulli success event with probability $p$, the expected waiting time of blind search is

$$
\mathbb{E}[T] = \frac{1}{p}.
$$

That is the paper's degenerate "heat-dumping" limit in executable form. We verify this by simulation.

In [ ]:
def simulate_geometric_trials(p: float, trials: int = 5000) -> np.ndarray:
    return np.random.geometric(p, size=trials)

ps = [1e-1, 1e-2, 1e-3, 1e-4]
rows = []
for p in ps:
    sims = simulate_geometric_trials(p)
    rows.append(
        {
            "p": p,
            "theory_mean": 1 / p,
            "sim_mean": sims.mean(),
            "sim_median": np.median(sims),
            "sim_std": sims.std(),
        }
    )
blind_df = pd.DataFrame(rows)
blind_df

In [ ]:
fig, ax = plt.subplots()
ax.plot(blind_df["p"], blind_df["theory_mean"], marker="o", label="theory")
ax.plot(blind_df["p"], blind_df["sim_mean"], marker="s", label="simulation")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("success probability p")
ax.set_ylabel("expected trials")
ax.set_title("Blind search waiting time: theory vs simulation")
ax.legend()
plt.show()

## 5. A toy folding landscape

To model the paper's structural claim — that successful search systems must exploit hidden geometry rather than flat enumeration — we compare two search modes on a bitstring landscape:

- **blind search**: sample uniformly until a target is found,
- **guided folding**: move downhill in Hamming distance to the target.

This is not a proof of cryptographic structure. It is a clean surrogate for the paper's folding imperative.

In [ ]:
def random_bitstring(n: int) -> np.ndarray:
    return np.random.randint(0, 2, size=n)

def hamming_distance(x: np.ndarray, y: np.ndarray) -> int:
    return int(np.sum(x != y))

def blind_search_target(target: np.ndarray, max_steps: int = 200000) -> int:
    n = len(target)
    for step in range(1, max_steps + 1):
        candidate = random_bitstring(n)
        if np.array_equal(candidate, target):
            return step
    return max_steps + 1

def guided_search_target(target: np.ndarray) -> int:
    x = random_bitstring(len(target))
    steps = 0
    while not np.array_equal(x, target):
        mismatch = np.where(x != target)[0]
        idx = np.random.choice(mismatch)
        x[idx] = target[idx]
        steps += 1
    return steps

n = 24
target = random_bitstring(n)

blind_trials = [blind_search_target(target, max_steps=200000) for _ in range(50)]
guided_trials = [guided_search_target(target) for _ in range(200)]

summary = pd.DataFrame(
    {
        "mode": ["blind", "guided"],
        "mean_steps": [np.mean(blind_trials), np.mean(guided_trials)],
        "median_steps": [np.median(blind_trials), np.median(guided_trials)],
        "min_steps": [np.min(blind_trials), np.min(guided_trials)],
        "max_steps": [np.max(blind_trials), np.max(guided_trials)],
    }
)
summary

In [ ]:
fig, ax = plt.subplots()
ax.hist(blind_trials, bins=15, alpha=0.7, label="blind")
ax.hist(guided_trials, bins=15, alpha=0.7, label="guided")
ax.set_xlabel("steps to target")
ax.set_ylabel("count")
ax.set_title("Blind search vs guided folding on a toy landscape")
ax.legend()
plt.show()

The gap above is the computational form of the paper's thermodynamic claim:

- flat search wastes budget,
- structural bias compresses the search,
- a "survivor basin" changes the operational class of the problem.

## 6. Universal runtime instantiated three ways

The paper's central isomorphism is that protein folding, Darwinian evolution, and cryptographic mining are all instances of the same compositional runtime.

Below we do not claim a full physical proof. We build transparent toy instantiations of that claim.

### 6.1 Protein-folding surrogate
Candidate states are bitstrings; fitness is negative Hamming distance to a native target.

### 6.2 Evolution surrogate
A population of candidate strings mutates and selection retains the fittest descendants.

### 6.3 Cryptographic-search surrogate
We vary a nonce in a real SHA-256 header prefix and audit a visible threshold such as leading zero bits.

In [ ]:
def selection_native(face: np.ndarray, target: np.ndarray) -> float:
    return -hamming_distance(face, target)

native_target = random_bitstring(32)

def protein_variation(x):
    y = x.copy()
    idx = np.random.randint(0, len(y))
    y[idx] = 1 - y[idx]
    return y

def protein_projection(y):
    return y

def protein_selection(face):
    return selection_native(face, native_target)

def retain_if_better(old, new, score):
    old_score = selection_native(old, native_target)
    return new if score >= old_score else old

protein_runtime = UniversalRuntime(
    variation=protein_variation,
    projection=protein_projection,
    selection=protein_selection,
    retain_rule=retain_if_better,
)

state = random_bitstring(32)
trajectory = []
for _ in range(150):
    out = protein_runtime.step(state)
    state = out["retained_state"]
    trajectory.append(-protein_selection(state))

fig, ax = plt.subplots()
ax.plot(trajectory)
ax.set_xlabel("step")
ax.set_ylabel("distance to native target")
ax.set_title("Protein-folding surrogate: retained survivor improves over time")
plt.show()

In [ ]:
def evolve_population(
    target: np.ndarray,
    pop_size: int = 80,
    generations: int = 80,
    mutation_rate: float = 0.04,
):
    n = len(target)
    population = np.random.randint(0, 2, size=(pop_size, n))
    best_scores = []

    for _ in range(generations):
        fitness = np.array([-hamming_distance(ind, target) for ind in population])
        best_scores.append(fitness.max())

        probs = np.exp(fitness - fitness.max())
        probs = probs / probs.sum()
        parents_idx = np.random.choice(np.arange(pop_size), size=pop_size, p=probs)
        new_pop = population[parents_idx].copy()

        mutation_mask = np.random.rand(pop_size, n) < mutation_rate
        new_pop = np.logical_xor(new_pop, mutation_mask).astype(int)
        population = new_pop

    return best_scores

evo_scores = evolve_population(random_bitstring(40))

fig, ax = plt.subplots()
ax.plot(evo_scores)
ax.set_xlabel("generation")
ax.set_ylabel("best fitness")
ax.set_title("Evolution surrogate: variation / selection / retained survivor")
plt.show()

In [ ]:
def leading_zero_bits(digest: bytes) -> int:
    bits = "".join(f"{b:08b}" for b in digest)
    return len(bits) - len(bits.lstrip("0"))

def sha256_nonce_scan(prefix: bytes, start: int = 0, count: int = 5000) -> pd.DataFrame:
    rows = []
    best = -1
    for nonce in range(start, start + count):
        msg = prefix + nonce.to_bytes(4, "big", signed=False)
        digest = hashlib.sha256(msg).digest()
        lz = leading_zero_bits(digest)
        best = max(best, lz)
        rows.append({"nonce": nonce, "leading_zero_bits": lz, "best_so_far": best})
    return pd.DataFrame(rows)

prefix = b"UniversalFoldingTheorem::demo::"
sha_scan = sha256_nonce_scan(prefix, count=5000)
sha_scan.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(sha_scan["nonce"], sha_scan["best_so_far"])
ax.set_xlabel("nonce")
ax.set_ylabel("best leading-zero-bit count so far")
ax.set_title("Real SHA-256 nonce scan (baseline search trace)")
plt.show()

pd.Series(sha_scan["leading_zero_bits"]).value_counts().sort_index().head(12)

### Interpretation

This SHA-256 section is intentionally conservative.

It demonstrates a **real search surface over real hashes**, but it does **not** claim that the notebook has extracted the full hidden geometry of SHA-256 or proven guided preimage navigation.

What it does show is:

- the cryptographic search can be written in the same variation / projection / selection language,
- blind threshold search has an explicit waiting-time profile,
- the paper's runtime vocabulary is executable on a real hash family.

## 7. Mark 1 attractor and local curvature loss

The paper's explicit geometric quantities are:

$$
H = \frac{\pi}{9},
\qquad
\epsilon(\theta) \approx \frac{\theta^2}{24},
\qquad
\epsilon(H)=\frac{H^2}{24}.
$$

We reproduce the local chord–arc picture numerically.

In [ ]:
def relative_curvature_loss(theta: np.ndarray) -> np.ndarray:
    return (theta - 2 * np.sin(theta / 2)) / theta

thetas = np.linspace(1e-4, 1.2, 1000)
exact_loss = relative_curvature_loss(thetas)
approx_loss = thetas**2 / 24

fig, ax = plt.subplots()
ax.plot(thetas, exact_loss, label="exact")
ax.plot(thetas, approx_loss, "--", label=r"$\theta^2/24$")
ax.axvline(H, color="black", linestyle=":", label=r"$H=\pi/9$")
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$\epsilon(\theta)$")
ax.set_title("Chord–arc curvature loss and the Mark 1 step")
ax.legend()
plt.show()

pd.DataFrame(
    {
        "H": [H],
        "epsilon_exact(H)": [relative_curvature_loss(np.array([H]))[0]],
        "epsilon_approx(H)": [H**2 / 24],
        "relative_error_of_approximation": [
            abs(relative_curvature_loss(np.array([H]))[0] - H**2/24) / relative_curvature_loss(np.array([H]))[0]
        ],
    }
)

## 8. A minimal AHRC-like controller

The paper describes AHRC as a convergence controller that pushes a drifting state back toward the Mark 1 band rather than as the fold itself. It explicitly gives a phase-error recurrence of the form

$$
\Delta_n = \text{state}_n - H_{\text{MARK1}} - \alpha \Delta_{n-1},
$$

and discusses off-band damping back toward equilibrium.

The cell below implements a **minimal transparent controller** inspired by that description:

$$
\Delta_n = x_n - H - \alpha \Delta_{n-1},
\qquad
x_{n+1} = x_n - \beta \Delta_n.
$$

This is a notebook surrogate, not a proof of the full AHRC formalism.

In [ ]:
def ahrc_like_sequence(x0: float, H: float, alpha: float = 0.25, beta: float = 0.55, steps: int = 40):
    xs = [x0]
    deltas = [x0 - H]
    for _ in range(steps - 1):
        delta = xs[-1] - H - alpha * deltas[-1]
        x_next = xs[-1] - beta * delta
        xs.append(x_next)
        deltas.append(delta)
    return np.array(xs), np.array(deltas)

starts = [0.05, 0.15, 0.55, 0.85]
fig, ax = plt.subplots()

for x0 in starts:
    xs, _ = ahrc_like_sequence(x0, H)
    ax.plot(xs, label=f"x0={x0:.2f}")

ax.axhline(H, color="black", linestyle="--", label=r"$H=\pi/9$")
ax.set_xlabel("iteration")
ax.set_ylabel("state")
ax.set_title("AHRC-like convergence toward the Mark 1 band")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for x0 in starts:
    _, deltas = ahrc_like_sequence(x0, H)
    ax.plot(np.abs(deltas), label=f"x0={x0:.2f}")

ax.set_yscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel(r"$|\Delta_n|$")
ax.set_title("AHRC-like error magnitude decay")
ax.legend()
plt.show()

## 9. Binding, Transformation, Readout as a triadic support condition

A recurring claim in the broader framework is that stable existence is not pairwise but triadic:

$$
\text{Existence} \cong \mathcal{B} \cap \mathcal{T} \cap \mathcal{R}.
$$

The paper also embeds the universal runtime into this triadic language.

A simple executable interpretation is:

- **Binding** = state retention,
- **Transformation** = lawful variation,
- **Readout** = projection plus audit.

If any one of these is missing, the runtime degenerates.

In [ ]:
triad_cases = pd.DataFrame(
    [
        ("Binding only", True, False, False, "frozen memory, no search"),
        ("Transformation only", False, True, False, "noise, no persistence"),
        ("Readout only", False, False, True, "inspection without state dynamics"),
        ("Binding + Transformation", True, True, False, "hidden process, no witness"),
        ("Binding + Readout", True, False, True, "static crystal / no adaptive movement"),
        ("Transformation + Readout", False, True, True, "volatile flux / no retained lineage"),
        ("Binding + Transformation + Readout", True, True, True, "stable searchable runtime"),
    ],
    columns=["case", "B", "T", "R", "interpretation"],
)
triad_cases

## 10. Notebook conclusions

This companion notebook reproduces and operationalizes the paper at three levels.

### Level A — exact quantities reproduced from the paper
- $H = \pi/9$,
- $\epsilon(H) = H^2/24$ as the local curvature-loss approximation,
- the universal search tuple,
- the finite-budget impossibility of flat exhaustive search.

### Level B — executable operational surrogates
- blind-search degeneracy,
- structure-exploiting folding on a toy landscape,
- a retained-survivor protein surrogate,
- a mutation-selection evolutionary surrogate,
- a real SHA-256 nonce-scan baseline,
- a minimal AHRC-like controller.

### Level C — what remains open
This notebook does **not** prove the full hidden geometric efficiency claim for real SHA-256 mining, nor the stronger cross-domain ontological claims of the paper.

What it does provide is a clean, runnable companion showing how the paper's runtime can be implemented, measured, and stress-tested in code.

In [ ]:
summary = {
    "H": H,
    "epsilon_H": epsilon_H,
    "blind_search_example_mean_steps": float(np.mean(blind_trials)),
    "guided_search_example_mean_steps": float(np.mean(guided_trials)),
    "sha_best_leading_zero_bits_in_demo": int(sha_scan["best_so_far"].max()),
}
summary